In [2]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import gdown
import zipfile
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="keras")

<h2 align="left" style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
Dataset Introduction
</font>
</h2>

<p dir="ltr" style="direction: ltr; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size="3">
The image dataset contains 44775 images, which includes training data and test samples. The images related to each person are located within their own directory. There is a variable number of images available for each person.
</font>
</p>

<p dir="ltr" style="direction: ltr; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size="3">
The features of the images related to the training data are located in a labeling file named <code>train.csv</code>, which contains 35332 rows.
There are 47 features in this file. Having a feature is indicated by the number +1, not having a feature is indicated by the number -1, and it being unknown whether a feature is present or not is specified by the number 0.
Descriptions of the important columns in this problem are provided in the table below.
</font>
</p>


<center>
<div dir=ltr style="direction: ltr;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
    
|Column|Description|
|:------:|:---:|
|Filename|Image name and directory|
|Identity|Unique number related to the person in the image|
|Male|Being male|
|Young|Being young|
|Middle_Aged|Being middle-aged|
|Senior|Being a senior|
    
</font>
</div>
</center>

<p dir=ltr style="direction: ltr; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font color="red"><b color='red'>Note:</b></font>
<font face="vazir" size=3>
Please be aware that a +1 in the <code>Male</code> column means being male, and a -1 in this column means being female.
</font>
</p>

<h2 align="left" style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
Reading the Dataset
</font>
</h2>

<p dir="ltr" style="direction: ltr; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
    Initially, you need to read the dataset files. The training features are stored in the <code>train.csv</code> file, and the directory related to the images of the test samples for which you must predict the category is stored in the <code>test.csv</code> file. If you deem it necessary, you can also separate a portion of the training data as validation data at your own discretion. Note that the target columns have been removed from the test data, and the purpose of this file's existence is to access the directory related to the test data images.
</font>
</p>


<p dir="ltr" style="direction: ltr; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
    <font face="vazir">
        <span style="color:orange"><b>Note:</b></span>
        The size of the training and test files is approximately 655 MB each. If loading these files in the Colab environment is difficult for you, you can download these two files directly in the Colab environment using the following command.
    </font>
</p>

In [ ]:
data_url = "https://drive.google.com/file/d/1_mGrKY786DV8OpT9Xw5gmxJcY0aq3Bb8/view?usp=share_link"
downloaded_file = gdown.download(data_url, output = None, quiet = False)
extract_folder = '.'
with zipfile.ZipFile(downloaded_file, 'r') as zip_ref:
        zip_ref.extractall(extract_folder)
os.remove(downloaded_file)

Downloading...
From (original): https://drive.google.com/uc?id=1_mGrKY786DV8OpT9Xw5gmxJcY0aq3Bb8
From (redirected): https://drive.google.com/uc?id=1_mGrKY786DV8OpT9Xw5gmxJcY0aq3Bb8&confirm=t&uuid=ba022cee-347a-42d5-a704-c31884c7f2bf
To: /home/norbert/Projects/computer_vision_oly/classification/her_face/data.zip
100%|██████████| 560M/560M [02:38<00:00, 3.54MB/s] 


In [4]:
train_data = pd.read_csv('data/train.csv') 
train_data.head()

,Filename,Identity,Male,Young,Middle_Aged,Senior,Asian,White,Black,Rosy_Cheeks,...,Big_Nose,Pointy_Nose,Heavy_Makeup,Wearing_Hat,Wearing_Earrings,Wearing_Necktie,Wearing_Lipstick,No_Eyewear,Eyeglasses,Attractive
0,n000003/0003_01.jpg,3,1,0,0,-1,-1,1,-1,-1,...,0,0,-1,-1,-1,0,-1,1,-1,-1
1,n000003/0009_02.jpg,3,1,-1,-1,1,-1,1,-1,-1,...,0,0,-1,-1,-1,0,-1,1,-1,-1
2,n000003/0010_01.jpg,3,1,0,0,-1,-1,1,-1,-1,...,0,0,-1,-1,-1,0,-1,1,-1,-1
3,n000003/0012_01.jpg,3,1,-1,-1,-1,-1,1,-1,-1,...,0,0,-1,-1,-1,0,-1,1,-1,-1
4,n000003/0014_01.jpg,3,1,0,0,-1,-1,1,-1,-1,...,0,0,-1,-1,-1,0,-1,1,-1,-1


In [5]:
test_data = pd.read_csv('data/test.csv')
test_data.head()

,Filename,Identity,Asian,White,Black,Rosy_Cheeks,Shiny_Skin,Bald,Wavy_Hair,Receding_Hairline,...,Big_Nose,Pointy_Nose,Heavy_Makeup,Wearing_Hat,Wearing_Earrings,Wearing_Necktie,Wearing_Lipstick,No_Eyewear,Eyeglasses,Attractive
0,n000002/0001_01.jpg,2,-1,1,-1,0,0,-1,1,-1,...,-1,1,1,-1,1,-1,1,1,-1,1
1,n000002/0012_01.jpg,2,-1,1,-1,0,0,-1,1,-1,...,-1,1,1,-1,1,-1,1,1,-1,1
2,n000002/0013_01.jpg,2,-1,1,-1,0,0,-1,1,-1,...,-1,1,1,-1,1,-1,1,1,-1,1
3,n000002/0018_04.jpg,2,-1,1,-1,-1,0,-1,1,-1,...,-1,1,1,-1,1,-1,1,1,-1,1
4,n000002/0020_01.jpg,2,-1,1,-1,0,1,-1,1,-1,...,-1,1,1,-1,1,-1,1,1,-1,1



<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
سپس نیاز است که تصاویر را از دایرکتوری های قرار گرفته در فایل های <code>test.csv</code> و <code>train.csv</code> بخوانید.
</font>
</p>

In [7]:
image_base_dir = 'data/image_data'
train_data['image_path'] = train_data['Filename'].apply(lambda x: os.path.join(image_base_dir, str(x)))
test_data['image_path'] = test_data['Filename'].apply(lambda x: os.path.join(image_base_dir, str(x)))

print("train_data : ", len(train_data))
print("test_data : ", len(test_data))

train_data :  35332
test_data :  8702


In [8]:
target_cols = ['Male', 'Young', 'Middle_Aged', 'Senior']
mapping = {1: 1.0, -1: 0.0, 0: 0.5}

for col in target_cols:
    train_data[col] = train_data[col].map(mapping)

train_df, val_df = train_test_split(train_data, test_size=0.1, random_state=42)

datagen = ImageDataGenerator(rescale=1./255)

train_generator = datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='image_path',
    y_col=target_cols,
    target_size=(128, 128),
    batch_size=32,
    class_mode='raw'
)

val_generator = datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col='image_path',
    y_col=target_cols,
    target_size=(128, 128),
    batch_size=32,
    class_mode='raw',
    shuffle=False
)

Found 31798 validated image filenames.
Found 3534 validated image filenames.


In [9]:
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(128, 128, 3))
base_model.trainable = False 

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.2)(x) 
x = Dense(128, activation='relu')(x)

predictions = Dense(4, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(optimizer=Adam(learning_rate=0.001), 
              loss='binary_crossentropy', 
              metrics=['accuracy'])

model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=5
)

I0000 00:00:1789812959.508642 3064401 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4035 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Epoch 1/5


I0000 00:00:1789812961.039556 3064401 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
I0000 00:00:1789812963.477504 3068890 service.cc:153] XLA service 0x788c54032f80 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1789812963.477531 3068890 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 4050 Laptop GPU, Compute Capability 8.9 (Driver: 13.0.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.22.0)
I0000 00:00:1789812963.563469 3068890 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1789812964.193513 3068890 cuda_dnn.cc:461] Loaded cuDNN version 92200
I0000 00:00:1789812964.257353 3068890 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_8148__.117
E0000 00:00:1789812967.144399 3068890 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup e

  5/994 ━━━━━━━━━━━━━━━━━━━━ 29s 30ms/step - accuracy: 0.5047 - loss: 0.7737 

I0000 00:00:1789812973.859456 3068890 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


620/994 ━━━━━━━━━━━━━━━━━━━━ 11s 30ms/step - accuracy: 0.8581 - loss: 0.3615

I0000 00:00:1789812993.401798 3068891 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_8148__.117
E0000 00:00:1789812996.220920 3068891 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


993/994 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.8654 - loss: 0.3481

E0000 00:00:1789813023.756311 3068892 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


994/994 ━━━━━━━━━━━━━━━━━━━━ 66s 54ms/step - accuracy: 0.8801 - loss: 0.3223 - val_accuracy: 0.9004 - val_loss: 0.2995
Epoch 2/5
994/994 ━━━━━━━━━━━━━━━━━━━━ 36s 36ms/step - accuracy: 0.8935 - loss: 0.2989 - val_accuracy: 0.9021 - val_loss: 0.2915
Epoch 3/5
994/994 ━━━━━━━━━━━━━━━━━━━━ 31s 31ms/step - accuracy: 0.8996 - loss: 0.2904 - val_accuracy: 0.9012 - val_loss: 0.2870
Epoch 4/5
994/994 ━━━━━━━━━━━━━━━━━━━━ 25s 25ms/step - accuracy: 0.9022 - loss: 0.2832 - val_accuracy: 0.9072 - val_loss: 0.2873
Epoch 5/5
994/994 ━━━━━━━━━━━━━━━━━━━━ 27s 27ms/step - accuracy: 0.9059 - loss: 0.2762 - val_accuracy: 0.9044 - val_loss: 0.2862


In [ ]:
val_generator.reset()
preds = model.predict(val_generator)

preds_for_f1 = np.where(preds >= 0.5, 1, 0)

y_true = val_generator.labels
y_true_for_f1 = np.where(y_true > 0.5, 1, 0)

f1 = f1_score(y_true_for_f1, preds_for_f1, average='macro')
print("Macro F1 Score on Validation set:", f1)

111/111 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step
Macro F1 Score on Validation set: 0.658570192680408


<h2 align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
 Prediction for Test Data and Output
</font>
</h2>

<p dir=ltr style="direction: ltr;text-align: left;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
    After feature engineering and modeling, you have an algorithm that can lead you from independent variables to target variables.
    <br>
    Use this model to predict the samples present in the test data and prepare the results in the format of the table (<code>dataframe</code>) below.
</font>
</p>

<div dir=ltr style="direction: ltr;text-align: left;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
    
|Column|Description|
|:------:|:---:|
|Male|Being male|
|Young|Being young|
|Middle_Aged|Being middle-aged|
|Senior|Being a senior|
    
</font>
</div>

<p dir=ltr style="direction: ltr;text-align: left;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
    The dataframe name must be <i>submission</i>; otherwise, the judging system cannot evaluate your attempt.
    <br>This dataframe only contains 4 columns related to the target columns and has 8702 rows.
    <br>
    For each row present in the <i>test</i> dataframe, you must have a predicted value.
    <br>
    The table below shows the first 5 rows of the <code>submission</code> dataframe. Of course, in your answer, the values of the <i>TripReason</i> column might be different.
</font>
</p>

<div style="text-align: center;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
    
| Male | Young | Middle_Aged | Senior |
|:----:|:-----:|:-----------:|:------:|
|  -1  |   1   |      -1     |    1   |
|  -1  |   1   |      -1     |    1   |
|  -1  |   1   |      -1     |    1   |
|  -1  |   1   |      -1     |    1   |
|  -1  |   1   |      -1     |    1   |

</font>
</div>

In [12]:
test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_data,
    x_col='image_path',
    y_col=None,
    target_size=(128, 128),
    batch_size=32,
    class_mode=None, 
    shuffle=False
)

test_preds = model.predict(test_generator)

preds_binary = np.where(test_preds >= 0.5, 1, -1)

submission = pd.DataFrame(preds_binary, columns=target_cols)

submission.head()

Found 8702 validated image filenames.
271/272 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step

E0000 00:00:1789813307.973526 3068893 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


272/272 ━━━━━━━━━━━━━━━━━━━━ 16s 51ms/step


,Male,Young,Middle_Aged,Senior
0,-1,1,-1,-1
1,-1,1,-1,-1
2,-1,1,-1,-1
3,-1,1,-1,-1
4,-1,1,-1,-1


In [ ]:
if not os.path.exists(os.path.join(os.getcwd(), 'AzSorateshMalomeh.ipynb')):
    %notebook -e AzSorateshMalomeh.ipynb

def compress(file_names):
    print("File Paths:")
    print(file_names)
    compression = zipfile.ZIP_DEFLATED
    with zipfile.ZipFile("result.zip", mode="w") as zf:
        for file_name in file_names:
            zf.write('./' + file_name, file_name, compress_type=compression)
            
submission.to_csv('submission.csv', index=False)
file_names = ['AzSorateshMalomeh.ipynb', 'submission.csv']
compress(file_names)

File Paths:
['AzSorateshMalomeh.ipynb', 'submission.csv']
